#**스마트폰 센서 데이터 기반 모션 분류**
# 단계3 : 단계별 모델링


## 0.미션

단계별로 나눠서 모델링을 수행하고자 합니다.  

* 단계1 : 정적(0), 동적(1) 행동 분류 모델 생성
* 단계2 : 세부 동작에 대한 분류모델 생성
    * 단계1 모델에서 0으로 예측 -> 정적 행동 3가지 분류 모델링
    * 단계1 모델에서 1으로 예측 -> 동적 행동 3가지 분류 모델링
* 모델 통합
    * 두 단계 모델을 통합하고, 새로운 데이터에 대해서 최종 예측결과와 성능평가가 나오도록 함수로 만들기
* 성능 비교
    * 기본 모델링의 성능과 비교
    * 모든 모델링은 [다양한 알고리즘 + 성능 튜닝]을 수행해야 합니다.


## 1.환경설정

### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
import joblib

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용

 <br/>  

* 세부 요구사항
    - data01_train.csv 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - data01_test.csv 를 불러와 'new_data' 이름으로 저장합니다.


In [ ]:
data = pd.read_csv('data01_train.csv')

In [ ]:
new_data = pd.read_csv('data01_test.csv')

## 2.데이터 전처리

* 세부 요구사항
    - Label 추가 : data 에 Activity_dynamic 를 추가합니다. Activity_dynamic은 과제1에서 is_dynamic과 동일한 값입니다.
    - x와 y1, y2로 분할하시오.
        * y1 : Activity
        * y2 : Activity_dynamic
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [ ]:
data['Activity'].unique()

array(['STANDING', 'LAYING', 'WALKING', 'WALKING_DOWNSTAIRS',
       'WALKING_UPSTAIRS', 'SITTING'], dtype=object)

In [ ]:
is_dynamic = {
    'STANDING': 0,
    'SITTING': 0,
    'LAYING': 0,
    'WALKING': 1,
    'WALKING_UPSTAIRS': 1,
    'WALKING_DOWNSTAIRS': 1
}

In [ ]:
data['Activity_dynamic'] = data['Activity'].map(is_dynamic)

In [ ]:
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity,Activity_dynamic
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING,0
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING,0
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING,0
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING,1
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS,1


In [ ]:
x = data.drop(['Activity', 'Activity_dynamic', 'subject'], axis=1)
y1 = data.loc[:, 'Activity'] # 개별 동작
y2 = data.loc[:, 'Activity_dynamic'] # 정적(0) vs 동적(1)

## **3.단계별 모델링**

![](https://github.com/DA4BAM/image/blob/main/step%20by%20step.png?raw=true)

## (1) 단계1 : 정적/동적 행동 분류 모델

* 세부 요구사항
    * 정적 행동(Laying, Sitting, Standing)과 동적 행동(동적 : Walking, Walking-Up, Walking-Down)을 구분하는 모델 생성.
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y2, test_size=0.2, random_state=42)

#### 1) 알고리즘1 : 랜덤 포레스트

In [ ]:
isDynamic_random_forest_model = RandomForestClassifier(random_state=42)

isDynamic_random_forest_model.fit(x_train, y_train)
y_pred = isDynamic_random_forest_model.predict(x_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 1.0
[[657   0]
 [  0 520]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       657
           1       1.00      1.00      1.00       520

    accuracy                           1.00      1177
   macro avg       1.00      1.00      1.00      1177
weighted avg       1.00      1.00      1.00      1177



In [ ]:
joblib.dump(isDynamic_random_forest_model, 'isDynamic_random_forest_model.pkl')

['isDynamic_random_forest_model.pkl']

#### 2) 알고리즘2 : XGBoost

In [ ]:
isDynamic_XGBoost_model = XGBClassifier(random_state=42)
isDynamic_XGBoost_model.fit(x_train, y_train)

y_pred = isDynamic_XGBoost_model.predict(x_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 1.0
[[657   0]
 [  0 520]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       657
           1       1.00      1.00      1.00       520

    accuracy                           1.00      1177
   macro avg       1.00      1.00      1.00      1177
weighted avg       1.00      1.00      1.00      1177



In [ ]:
joblib.dump(isDynamic_XGBoost_model, 'isDynamic_XGBoost_model.pkl')

['isDynamic_XGBoost_model.pkl']

## (2) 단계2-1 : 정적 동작 세부 분류

* 세부 요구사항
    * 정적 행동(Laying, Sitting, Standing)인 데이터 추출
    * Laying, Sitting, Standing 를 분류하는 모델을 생성
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [ ]:
static_activity = (data['Activity']=='LAYING') | (data['Activity']=='SITTING') | (data['Activity']=='STANDING')
static_data = data[static_activity]

In [ ]:
x = static_data.drop(['subject', 'Activity', 'Activity_dynamic'], axis=1)
y = static_data.loc[:, 'Activity']

In [ ]:
static_data['Activity'].unique()

array(['STANDING', 'LAYING', 'SITTING'], dtype=object)

#### 1) 알고리즘1 : 랜덤 포레스트

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
static_random_forest_model = RandomForestClassifier(random_state=42)

static_random_forest_model.fit(x_train, y_train)
y_pred = static_random_forest_model.predict(x_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.973724884080371
[[221   0   0]
 [  0 199   6]
 [  0  11 210]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       221
     SITTING       0.95      0.97      0.96       205
    STANDING       0.97      0.95      0.96       221

    accuracy                           0.97       647
   macro avg       0.97      0.97      0.97       647
weighted avg       0.97      0.97      0.97       647



In [ ]:
joblib.dump(static_random_forest_model, 'static_random_forest_model.pkl')

['static_random_forest_model.pkl']

#### 1) 알고리즘2 : XGBoost

In [ ]:
# XGBoost는 label로 숫자를 원함.
static_le = LabelEncoder()
y_xgb = static_le.fit_transform(y)

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y_xgb, test_size=0.2, random_state=42)

In [ ]:
static_XGBoost_model = XGBClassifier(random_state=42)
static_XGBoost_model.fit(x_train, y_train)

y_pred = static_XGBoost_model.predict(x_val)

y_val = static_le.inverse_transform(y_val)
y_pred = static_le.inverse_transform(y_pred)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9829984544049459
[[221   0   0]
 [  0 201   4]
 [  0   7 214]]
              precision    recall  f1-score   support

      LAYING       1.00      1.00      1.00       221
     SITTING       0.97      0.98      0.97       205
    STANDING       0.98      0.97      0.97       221

    accuracy                           0.98       647
   macro avg       0.98      0.98      0.98       647
weighted avg       0.98      0.98      0.98       647



In [ ]:
joblib.dump(static_XGBoost_model, 'static_XGBoost_model.pkl')

['static_XGBoost_model.pkl']

## (3) 단계2-2 : 동적 동작 세부 분류

* 세부 요구사항
    * 동동적 행동(Walking, Walking Upstairs, Walking Downstairs)인 데이터 추출
    * Walking, Walking Upstairs, Walking Downstairs 를 분류하는 모델을 생성
    * 몇가지 모델을 만들고 가장 성능이 좋은 모델을 선정하시오.

In [ ]:
dynamic_activity = (data['Activity']=='WALKING') | (data['Activity']=='WALKING_UPSTAIRS') | (data['Activity']=='WALKING_DOWNSTAIRS')
dynamic_data = data[dynamic_activity]

In [ ]:
x = dynamic_data.drop(['subject', 'Activity', 'Activity_dynamic'], axis=1)
y = dynamic_data.loc[:, 'Activity']

In [ ]:
dynamic_data['Activity'].unique()

array(['WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS'], dtype=object)

#### 1) 알고리즘1 : 랜덤 포레스트

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
dynamic_random_forest_model = RandomForestClassifier(random_state=42)

dynamic_random_forest_model.fit(x_train, y_train)
y_pred = dynamic_random_forest_model.predict(x_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9886792452830189
[[196   0   0]
 [  2 159   2]
 [  1   1 169]]
                    precision    recall  f1-score   support

           WALKING       0.98      1.00      0.99       196
WALKING_DOWNSTAIRS       0.99      0.98      0.98       163
  WALKING_UPSTAIRS       0.99      0.99      0.99       171

          accuracy                           0.99       530
         macro avg       0.99      0.99      0.99       530
      weighted avg       0.99      0.99      0.99       530



In [ ]:
joblib.dump(dynamic_random_forest_model, 'dynamic_random_forest_model.pkl')

['dynamic_random_forest_model.pkl']

#### 1) 알고리즘2 : XGBoost

In [ ]:
# XGBoost는 label로 숫자를 원함.
dynamic_le = LabelEncoder()
y_xgb = dynamic_le.fit_transform(y)

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y_xgb, test_size=0.2, random_state=42)

In [ ]:
dynamic_XGBoost_model = XGBClassifier(random_state=42)
dynamic_XGBoost_model.fit(x_train, y_train)

y_pred = dynamic_XGBoost_model.predict(x_val)

y_val = dynamic_le.inverse_transform(y_val)
y_pred = dynamic_le.inverse_transform(y_pred)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9924528301886792
[[195   1   0]
 [  0 161   2]
 [  1   0 170]]
                    precision    recall  f1-score   support

           WALKING       0.99      0.99      0.99       196
WALKING_DOWNSTAIRS       0.99      0.99      0.99       163
  WALKING_UPSTAIRS       0.99      0.99      0.99       171

          accuracy                           0.99       530
         macro avg       0.99      0.99      0.99       530
      weighted avg       0.99      0.99      0.99       530



In [ ]:
joblib.dump(dynamic_XGBoost_model, 'dynamic_XGBoost_model.pkl')

['dynamic_XGBoost_model.pkl']

## (4) 분류 모델 합치기


* 세부 요구사항
    * 두 단계 모델을 통합하고, 새로운 데이터(test)에 대해서 최종 예측결과와 성능평가가 나오도록 함수로 만들기
    * 데이터 파이프라인 구축 : test데이터가 로딩되어 전처리 과정을 거치고, 예측 및 성능 평가 수행

![](https://github.com/DA4BAM/image/blob/main/pipeline%20function.png?raw=true)

#### 1) 함수 만들기

In [ ]:
def activity_classifier_model(test_data):
    final_predict = []

    test_data.dropna(inplace=True)
    test_data.reset_index(drop=True, inplace=True)

    X_test = test_data.drop(['subject', 'Activity'], axis=1)
    y_test = test_data.loc[:, 'Activity']

    isDynamic_pred = isDynamic_random_forest_model.predict(X_test)

    for idx in range(len(isDynamic_pred)):
        pred = None
        if(isDynamic_pred[idx] == 1):
            pred = dynamic_random_forest_model.predict(X_test.iloc[[idx], :])
        else:
            pred = static_random_forest_model.predict(X_test.iloc[[idx], :])
        final_predict.append(pred[0])

    print('accuracy :',accuracy_score(y_test, final_predict))
    print('='*60)
    print(confusion_matrix(y_test, final_predict))
    print('='*60)
    print(classification_report(y_test, final_predict))

    return pd.Series(final_predict, name='predict_result')

In [ ]:
test_data = pd.read_csv('data01_test.csv')

In [ ]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1471 entries, 0 to 1470
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 6.3+ MB


In [ ]:
test_data.shape

(1471, 563)

In [ ]:
test_data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.284379,-0.021981,-0.116683,-0.992490,-0.979640,-0.963321,-0.992563,-0.977304,-0.958142,-0.938850,...,-0.850065,-0.018043,0.092304,0.074220,-0.714534,-0.671943,-0.018351,-0.185733,22,SITTING
1,0.277440,-0.028086,-0.118412,-0.996620,-0.927676,-0.972294,-0.997346,-0.931405,-0.971788,-0.939837,...,-0.613367,-0.022456,-0.155414,0.247498,-0.112257,-0.826816,0.184489,-0.068699,15,STANDING
2,0.305833,-0.041023,-0.087303,0.006880,0.182800,-0.237984,0.005642,0.028616,-0.236474,0.016311,...,0.394388,-0.362616,0.171069,0.576349,-0.688314,-0.743234,0.272186,0.053101,22,WALKING
3,0.276053,-0.016487,-0.108381,-0.995379,-0.983978,-0.975854,-0.995877,-0.985280,-0.974907,-0.941425,...,-0.841455,0.289548,0.079801,-0.020033,0.291898,-0.639435,-0.111998,-0.123298,8,SITTING
4,0.271998,0.016904,-0.078856,-0.973468,-0.702462,-0.869450,-0.979810,-0.711601,-0.856807,-0.920760,...,0.214219,0.010111,0.114179,-0.830776,-0.325098,-0.840817,0.116237,-0.096615,5,STANDING


In [ ]:
predict_result = activity_classifier_model(test_data)

accuracy : 0.9741672331747111
[[292   0   0   0   0   0]
 [  0 244  10   0   0   0]
 [  0  17 270   0   0   0]
 [  0   0   0 224   0   4]
 [  0   0   0   1 192   2]
 [  0   0   0   1   3 211]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       292
           SITTING       0.93      0.96      0.95       254
          STANDING       0.96      0.94      0.95       287
           WALKING       0.99      0.98      0.99       228
WALKING_DOWNSTAIRS       0.98      0.98      0.98       195
  WALKING_UPSTAIRS       0.97      0.98      0.98       215

          accuracy                           0.97      1471
         macro avg       0.97      0.97      0.97      1471
      weighted avg       0.97      0.97      0.97      1471



In [ ]:
predict_result

,predict_result
0,SITTING
1,STANDING
2,WALKING
3,SITTING
4,STANDING
...,...
1466,SITTING
1467,STANDING
1468,WALKING_DOWNSTAIRS
1469,LAYING


## 모델 저장 후 모델 불러와서 결합

In [ ]:
isDynamic_model = joblib.load('isDynamic_XGBoost_model.pkl')
static_model = joblib.load('static_XGBoost_model.pkl')
dynamic_model = joblib.load('dynamic_XGBoost_model.pkl')

In [ ]:
def activity_classifier_model_usingSaveModel(test_data):
    final_predict = []

    test_data.dropna(inplace=True)
    test_data.reset_index(drop=True, inplace=True)

    X_test = test_data.drop(['subject', 'Activity'], axis=1)
    y_test = test_data.loc[:, 'Activity']

    isDynamic_pred = isDynamic_model.predict(X_test)

    for idx in range(len(isDynamic_pred)):
        if(isDynamic_pred[idx] == 1):
            pred = dynamic_model.predict(X_test.iloc[[idx], :])
            if isinstance(pred[0], (int, np.integer)):
                pred = dynamic_le.inverse_transform(pred)
        else:
            pred = static_model.predict(X_test.iloc[[idx], :])
            if isinstance(pred[0], (int, np.integer)):
                pred = static_le.inverse_transform(pred)

        final_predict.append(pred[0])

    print('accuracy :',accuracy_score(y_test, final_predict))
    print('='*60)
    print(confusion_matrix(y_test, final_predict))
    print('='*60)
    print(classification_report(y_test, final_predict))

    return pd.Series(final_predict, name='predict_result')

In [ ]:
predict_result = activity_classifier_model_usingSaveModel(test_data)

accuracy : 0.9843643779741672
[[292   0   0   0   0   0]
 [  0 244  10   0   0   0]
 [  0   6 281   0   0   0]
 [  0   0   0 223   0   5]
 [  0   0   0   1 194   0]
 [  0   0   0   1   0 214]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       292
           SITTING       0.98      0.96      0.97       254
          STANDING       0.97      0.98      0.97       287
           WALKING       0.99      0.98      0.98       228
WALKING_DOWNSTAIRS       1.00      0.99      1.00       195
  WALKING_UPSTAIRS       0.98      1.00      0.99       215

          accuracy                           0.98      1471
         macro avg       0.98      0.98      0.98      1471
      weighted avg       0.98      0.98      0.98      1471



In [ ]:
predict_result

,predict_result
0,SITTING
1,STANDING
2,WALKING
3,SITTING
4,STANDING
...,...
1466,SITTING
1467,STANDING
1468,WALKING_DOWNSTAIRS
1469,LAYING
